In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm import tqdm

# ----------------------------
# Config
# ----------------------------
START_DATE = "2005-01-01"
END_DATE   = "2025-01-01"

# Index we use for "market pressure"
INDEX_TICKER = "^NSEI"   # Nifty 50

# You can replace/extend this with Nifty 500.
# Easiest: keep a CSV with a 'Symbol' column and read it here.
# For now I put a seed list – extend this to NIFTY 50/100/500 as needed.
SEED_TICKERS = [
    "RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "AXISBANK.NS", "KOTAKBANK.NS",
    "SBIN.NS", "INFY.NS", "TCS.NS", "HINDUNILVR.NS", "LT.NS",
]

def load_nifty500_tickers_from_csv(path="/content/drive/MyDrive/nifty100_like_100_tickers.csv"):
    """
    Optional: read tickers from a CSV you upload.

    Accepts:
      - Column 'Symbol' like RELIANCE, TCS (will append .NS)
      - OR column 'ticker' like RELIANCE.NS, TCS.NS
      - OR first column if neither exists
    """
    try:
        df = pd.read_csv(path)

        if "ticker" in df.columns:
            vals = df["ticker"].astype(str).str.strip().tolist()
            tickers = []
            for v in vals:
                if not v or v.lower() == "nan":
                    continue
                tickers.append(v if v.endswith(".NS") or v.startswith("^") else v + ".NS")
            print(f"Loaded {len(tickers)} tickers from {path} (ticker column)")
            return tickers

        if "Symbol" in df.columns:
            syms = df["Symbol"].astype(str).str.strip().tolist()
            tickers = [s if s.endswith(".NS") else s + ".NS" for s in syms if s and s.lower() != "nan"]
            print(f"Loaded {len(tickers)} tickers from {path} (Symbol column)")
            return tickers

        # fallback: first column
        vals = df.iloc[:, 0].astype(str).str.strip().tolist()
        tickers = []
        for v in vals:
            if not v or v.lower() == "nan":
                continue
            tickers.append(v if v.endswith(".NS") or v.startswith("^") else v + ".NS")
        print(f"Loaded {len(tickers)} tickers from {path} (first column)")
        return tickers

    except Exception as e:
        print(f"Could not load {path}: {e}")
        print("Using SEED_TICKERS instead.")
        return SEED_TICKERS


# CHOOSE: either use CSV or the seed list
TICKERS = load_nifty500_tickers_from_csv()  # or: TICKERS = SEED_TICKERS

# ----------------------------
# Helper functions
# ----------------------------

def _flatten_yf_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    yfinance may return MultiIndex columns (or tuples).
    Convert them into plain strings so downstream renaming never crashes.
    """
    if df is None or df.empty:
        return df

    # MultiIndex columns
    if hasattr(df.columns, "nlevels") and getattr(df.columns, "nlevels", 1) > 1:
        df = df.copy()
        df.columns = [
            "_".join([str(x) for x in col if x not in (None, "", " ")])
            for col in df.columns
        ]
        return df

    # Tuple columns
    df = df.copy()
    new_cols = []
    for c in df.columns:
        if isinstance(c, tuple):
            new_cols.append("_".join(map(str, c)))
        else:
            new_cols.append(str(c))
    df.columns = new_cols
    return df


def download_ohlcv(ticker, start=START_DATE, end=END_DATE):
    """Download OHLCV from yfinance and standardize column names."""
    df = yf.download(
        ticker,
        start=start,
        end=end,
        progress=False,
        auto_adjust=False,
        group_by="column"
    )
    if df is None or df.empty:
        return pd.DataFrame()

    # Flatten tuple/MultiIndex columns to strings
    df = _flatten_yf_columns(df)

    # --- FIX: normalize columns like "Open_^NSEI" -> "open" ---
    def normalize_col(c: str) -> str:
        s = str(c).strip().lower()

        # If we got "open_^nsei" / "close_reliance.ns" etc., keep only the left part
        # before the first underscore.
        base = s.split("_", 1)[0]

        # Some variants can appear
        base = base.replace("adj close", "adj_close").replace("adjclose", "adj_close")

        return base

    df = df.rename(columns={c: normalize_col(c) for c in df.columns})

    # If both close + adj_close exist, prefer close (or keep both if you want)
    # Here we keep just close as before.
    if "adj_close" in df.columns and "close" not in df.columns:
        df["close"] = df["adj_close"]

    needed = ["open", "high", "low", "close", "volume"]
    missing = [n for n in needed if n not in df.columns]
    if missing:
        print(f"[WARN] {ticker}: missing cols {missing}, skipping.")
        return pd.DataFrame()

    df = df[needed].copy()
    df.index.name = "date"
    return df


def add_stock_features(df):
    """
    Add price-based features similar to base paper, causally:
    - logret (daily log return)
    - MA10, MA50, MA100 of close
    - volatility windows
    - volume z-score
    - trend_10_50 (short vs medium MA)
    """
    df = df.copy()
    df["logret"] = np.log(df["close"] / df["close"].shift(1))

    df["ma10"] = df["close"].rolling(10).mean()
    df["ma50"] = df["close"].rolling(50).mean()
    df["ma100"] = df["close"].rolling(100).mean()

    df["vol_10"] = df["logret"].rolling(10).std()
    df["vol_20"] = df["logret"].rolling(20).std()
    df["vol_60"] = df["logret"].rolling(60).std()

    vol_ma = df["volume"].rolling(20).mean()
    vol_std = df["volume"].rolling(20).std()
    df["volume_z"] = (df["volume"] - vol_ma) / vol_std

    # Short vs medium term trend (like "MA for 10 vs 50 days")
    df["trend_10_50"] = (df["ma10"] - df["ma50"]) / df["ma50"]

    # Drop initial NaNs
    df = df.dropna()
    return df


def add_index_features(index_ticker=INDEX_TICKER):
    """Build index features: daily log return + 10-day volatility."""
    idx_df = download_ohlcv(index_ticker)
    if idx_df.empty:
        raise ValueError("Index download failed.")

    idx_df["logret"] = np.log(idx_df["close"] / idx_df["close"].shift(1))
    idx_df["vol_10"] = idx_df["logret"].rolling(10).std()
    idx_df = idx_df.dropna()
    idx_df = idx_df.rename(
        columns={
            "logret": "idx_logret_1d",
            "vol_10": "idx_vol_10",
        }
    )
    return idx_df[["idx_logret_1d", "idx_vol_10"]]


def build_per_stock_frames(
    tickers,
    big_move_quantile=0.9,
    min_len=300,
):
    """
    For each ticker:
    - download OHLCV
    - add stock features
    - merge index features
    - compute logret_next and global big_move_next label
    Returns:
      stock_frames: dict[ticker] -> DataFrame(features + big_move_next)
      big_move_thr: scalar threshold used for |logret_next|
    """
    idx_feat = add_index_features()

    frames = {}
    all_logret_next = []

    for ticker in tqdm(tickers):
        base = download_ohlcv(ticker)
        if base.empty or len(base) < min_len:
            continue

        feat = add_stock_features(base)
        merged = feat.join(idx_feat, how="inner")
        if merged.empty or len(merged) < min_len:
            continue

        # Forward log return (next day) for the LABEL
        merged["logret_next"] = merged["logret"].shift(-1)

        # Drop last row where logret_next is NaN
        merged = merged.dropna(subset=["logret_next"])

        if len(merged) < min_len:
            continue

        frames[ticker] = merged
        all_logret_next.append(merged["logret_next"])

    if not frames:
        raise ValueError("No valid stocks downloaded. Check tickers/date range.")

    all_logret_next = pd.concat(all_logret_next)
    big_thr = all_logret_next.abs().quantile(big_move_quantile)
    print(f"Global big-move threshold (|logret_next|): {big_thr:.6f}")

    # Now assign labels per stock
    for ticker, df in frames.items():
        df["big_move_next"] = (df["logret_next"].abs() >= big_thr).astype(int)

    return frames, big_thr


# ---- Run Cell 1 ----
stock_frames, global_big_move_thr = build_per_stock_frames(TICKERS)

print(f"Built per-stock frames for {len(stock_frames)} tickers.")
for t, df in list(stock_frames.items())[:3]:
    pos_rate = df["big_move_next"].mean()
    print(f"{t}: rows={len(df)}, big-move rate={pos_rate:.3f}")


Loaded 100 tickers from /content/drive/MyDrive/nifty100_like_100_tickers.csv (ticker column)


 23%|██▎       | 23/100 [00:07<00:22,  3.42it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TATAMOTORS.NS']: YFTzMissingError('possibly delisted; no timezone found')
 75%|███████▌  | 75/100 [00:21<00:07,  3.56it/s]ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ZOMATO.NS']: YFTzMissingError('possibly delisted; no timezone found')
100%|██████████| 100/100 [00:29<00:00,  3.41it/s]

Global big-move threshold (|logret_next|): 0.033886
Built per-stock frames for 97 tickers.
RELIANCE.NS: rows=4227, big-move rate=0.074
TCS.NS: rows=4227, big-move rate=0.064
HDFCBANK.NS: rows=4227, big-move rate=0.059


In [ ]:
# ==== RISK_DATA_v5 – Cell 2 (RAM efficient): streaming windows + streaming scaler ====

import numpy as np
import tensorflow as tf

T = 120

FEATURE_COLS = [
    "logret",
    "ma10", "ma50", "ma100",
    "vol_10", "vol_20", "vol_60",
    "volume_z",
    "trend_10_50",
    "idx_logret_1d",
    "idx_vol_10",
]

LABEL_COL = "big_move_next"

BATCH_SIZE = 256

# Split by time INSIDE each stock (keeps temporal integrity)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9


def iter_windows_for_stock(df, split: str, T=T, feature_cols=FEATURE_COLS, label_col=LABEL_COL):
    """
    Yields (X_window, y) for a single stock df for the requested split.
    Window X is shape (T, F). y is scalar.
    """
    feats = df[feature_cols].to_numpy(dtype=np.float32, copy=False)
    labels = df[label_col].to_numpy(dtype=np.int8, copy=False)

    n = len(df)
    if n <= T + 1:
        return

    # Determine split boundaries in terms of "label index" i (where label corresponds to row i)
    # windows use rows [i-T, i)
    i_start = T
    i_end = n  # exclusive for i, since y uses labels[i]

    total = i_end - i_start
    tr_end = i_start + int(total * TRAIN_FRAC)
    va_end = tr_end + int(total * VAL_FRAC)

    if split == "train":
        lo, hi = i_start, tr_end
    elif split == "val":
        lo, hi = tr_end, va_end
    elif split == "test":
        lo, hi = va_end, i_end
    else:
        raise ValueError("split must be train/val/test")

    for i in range(lo, hi):
        X = feats[i - T:i]    # (T, F)
        y = labels[i]         # scalar
        yield X, y


def iter_windows_all_stocks(stock_frames, split: str):
    """Yields windows across all stocks for a given split."""
    for t, df in stock_frames.items():
        yield from iter_windows_for_stock(df, split=split)


# --------- Streaming scaler (Welford) ----------
class StreamingStandardizer:
    def __init__(self, F):
        self.F = F
        self.count = 0
        self.mean = np.zeros(F, dtype=np.float64)
        self.M2 = np.zeros(F, dtype=np.float64)

    def update_batch(self, X_batch_2d):
        # X_batch_2d: (N, F)
        X = X_batch_2d.astype(np.float64, copy=False)
        for x in X:
            self.count += 1
            delta = x - self.mean
            self.mean += delta / self.count
            delta2 = x - self.mean
            self.M2 += delta * delta2

    def finalize(self):
        var = self.M2 / max(self.count - 1, 1)
        std = np.sqrt(var)
        std[std == 0] = 1.0
        return self.mean.astype(np.float32), std.astype(np.float32)


print("Computing scaler stats from TRAIN split (streaming, RAM-safe)...")

F = len(FEATURE_COLS)
scaler = StreamingStandardizer(F)

# Update scaler using train windows only (flatten each window to (T, F) rows)
# To keep it fast, update in chunks.
chunk = []
CHUNK_MAX_ROWS = 20000  # rows of (F) before updating

for X, _y in iter_windows_all_stocks(stock_frames, split="train"):
    chunk.append(X)  # X is (T,F)
    if len(chunk) >= 64:  # 64 windows * 120 rows = 7680 rows
        Xc = np.concatenate(chunk, axis=0)  # (64*T, F)
        scaler.update_batch(Xc)
        chunk = []

# flush
if chunk:
    Xc = np.concatenate(chunk, axis=0)
    scaler.update_batch(Xc)

mean, std = scaler.finalize()
print("Scaler fitted. count rows:", scaler.count)

# --------- tf.data pipelines ----------
def make_tf_dataset(stock_frames, split: str, mean, std, batch_size=BATCH_SIZE, shuffle=False):
    output_signature = (
        tf.TensorSpec(shape=(T, F), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int8),
    )

    def gen():
        for X, y in iter_windows_all_stocks(stock_frames, split=split):
            # standardize on the fly
            Xs = (X - mean) / std
            yield Xs, y

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    if shuffle:
        ds = ds.shuffle(20000, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size)

    # Add repeat only for training
    if split == "train":
      ds = ds.repeat()

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds



train_ds = make_tf_dataset(stock_frames, "train", mean, std, shuffle=True)
val_ds   = make_tf_dataset(stock_frames, "val",   mean, std, shuffle=False)
test_ds  = make_tf_dataset(stock_frames, "test",  mean, std, shuffle=False)

print("Datasets ready:")
print("train_ds:", train_ds)
print("val_ds:", val_ds)
print("test_ds:", test_ds)

# Optional: quick sanity check batch
for xb, yb in train_ds.take(1):
    print("Batch X:", xb.shape, xb.dtype, "Batch y:", yb.shape, yb.dtype)
    print("Pos rate (batch):", tf.reduce_mean(tf.cast(yb, tf.float32)).numpy())


Computing scaler stats from TRAIN split (streaming, RAM-safe)...
Scaler fitted. count rows: 29286960
Datasets ready:
train_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 120, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int8, name=None))>
val_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 120, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int8, name=None))>
test_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 120, 11), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int8, name=None))>
Batch X: (256, 120, 11) <dtype: 'float32'> Batch y: (256,) <dtype: 'int8'>
Pos rate (batch): 0.08203125


In [ ]:
# ==== RISK_MSF_v5 – Cell 3: Build models, train, evaluate (tf.data streaming) ====

def count_windows(stock_frames, split="train"):
    return sum(1 for _ in iter_windows_all_stocks(stock_frames, split=split))

NUM_TRAIN_WINDOWS = count_windows(stock_frames, "train")
NUM_VAL_WINDOWS   = count_windows(stock_frames, "val")

steps_per_epoch     = NUM_TRAIN_WINDOWS // BATCH_SIZE
validation_steps    = NUM_VAL_WINDOWS // BATCH_SIZE


import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

tf.keras.backend.clear_session()

print("TF version:", tf.__version__)

# Safety checks
for name in ["train_ds", "val_ds", "test_ds"]:
    if name not in globals():
        raise RuntimeError(f"{name} not found. Run Cell 2 first (streaming version).")

# Infer input shape from a batch
xb, yb = next(iter(train_ds))
input_shape = (xb.shape[1], xb.shape[2])
print("Input shape:", input_shape)

def build_lstm_model(input_shape, l2=1e-4, dropout=0.2):
    inp = layers.Input(shape=input_shape)
    x = layers.LSTM(64, return_sequences=True, kernel_regularizer=regularizers.l2(l2))(inp)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(32, kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inp, out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model

model = build_lstm_model(input_shape)
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=5, mode="max", restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", patience=2, mode="max", factor=0.5, min_lr=1e-5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    batch_size=None,   # IMPORTANT: batching is already done in tf.data
    callbacks=callbacks,
    verbose=1
)

test_metrics = model.evaluate(test_ds, verbose=0)
print(dict(zip(model.metrics_names, test_metrics)))


TF version: 2.19.0
Input shape: (120, 11)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 120, 11)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 120, 64)        │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 120, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,961 (128.75 KB)

 Trainable params: 32,961 (128.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
953/953 ━━━━━━━━━━━━━━━━━━━━ 120s 117ms/step - auc: 0.6828 - loss: 0.3133 - precision: 0.2888 - recall: 0.0102 - val_auc: 0.7139 - val_loss: 0.3382 - val_precision: 0.5771 - val_recall: 0.0532 - learning_rate: 0.0010
Epoch 2/30
953/953 ━━━━━━━━━━━━━━━━━━━━ 137s 144ms/step - auc: 0.7326 - loss: 0.2790 - precision: 0.5848 - recall: 0.0421 - val_auc: 0.7175 - val_loss: 0.3356 - val_precision: 0.5926 - val_recall: 0.0720 - learning_rate: 0.0010
Epoch 3/30
953/953 ━━━━━━━━━━━━━━━━━━━━ 110s 115ms/step - auc: 0.7357 - loss: 0.2785 - precision: 0.5729 - recall: 0.0435 - val_auc: 0.7203 - val_loss: 0.3333 - val_precision: 0.6271 - val_recall: 0.0473 - learning_rate: 0.0010
Epoch 4/30
953/953 ━━━━━━━━━━━━━━━━━━━━ 110s 115ms/step - auc: 0.7358 - loss: 0.2767 - precision: 0.5842 - recall: 0.0433 - val_auc: 0.7164 - val_loss: 0.3335 - val_precision: 0.6052 - val_recall: 0.0666 - learning_rate: 0.0010
Epoch 5/30
953/953 ━━━━━━━━━━━━━━━━━━━━ 110s 116ms/step - auc: 0.7398 - loss: 0.2767 - p

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [ ]:
test_metrics = model.evaluate(test_ds, verbose=0)
print(dict(zip(model.metrics_names, test_metrics)))


{'loss': 0.19958649575710297, 'compile_metrics': 0.6950035095214844}


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
